In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
let divide (numerator: float) (denominator: float) =
    if denominator = 0.0 then failwith "Division by zero"
    numerator / denominator

# An Interpreter for a Simple Programming Language

This F# edition supplies a complete worked solution after the exercise prompts so that every cell is executable. Compare the typed implementation with your own solution.

In this notebook we develop an interpreter for a small programming language.

In [ ]:
#load "Interpreter.fsx"
open LanguageSyntax
open System.Numerics

type IO = { Read: unit -> Value; Write: Value list -> unit }
type State =
    { Values: Map<string, Value>
      Definitions: Map<string, string list * Statement> }
exception ReturnValue of Value
let emptyState = { Values = Map.empty; Definitions = Map.empty }

let truth = function
    | Boolean b -> b
    | Integer n -> n <> BigInteger.Zero
    | Real n -> n <> 0.0
    | Text s -> s <> ""
// Numeric comparison and floor-quotient rules have one canonical implementation.
let asNumber = Interpreter.asNumber
let compareIntegerReal = Interpreter.compareIntegerReal
let compareValues = Interpreter.compareValues
let equalValues = Interpreter.equalValues
let floorDivide = Interpreter.floorDivide
let floatModulo = Interpreter.floatModulo

let arithmetic op left right =
    match op, left, right with
    | Add, Text a, Text b -> Text(a + b)
    | _, Integer a, Integer b ->
        match op with
        | Add -> Integer(a + b) | Subtract -> Integer(a - b) | Multiply -> Integer(a * b)
        | Divide -> Integer(floorDivide a b)
        | Modulo -> Integer(a - floorDivide a b * b)
        | _ -> failwith "Expected an arithmetic operator"
    | _ ->
        let a, b = asNumber left, asNumber right
        match op with
        | Add -> Real(a + b) | Subtract -> Real(a - b) | Multiply -> Real(a * b)
        | Divide -> Real(divide a b) | Modulo -> Real(floatModulo a b)
        | _ -> failwith "Expected an arithmetic operator"

let mathFunctions = Map.ofList ["sqrt", sqrt; "exp", exp; "ln", log; "sin", sin; "cos", cos; "tan", tan; "arctan", atan]

The scanner is generated from `LanguageSyntax/Lexer.fsl`. The exercise syntax extends the simple interpreter with strings, function definitions, `return`, `for`, and boolean connectives.

We allow both *single-line comments* and *multi-line comments*.
- The regular expression `/\*(.|\n)*?\*/` recognizes multi-line comments.
  Multi-line comments start with the string `/*` and end with the string `*/`.
  Note the use of the *non-greedy* quantor `*?`.  If we have code like
  ```
  /* blah */ a := 1; /* blub */
  ```
  the greedy quantor would recognize the whole line as one comment. 
- The regular expression `//.*` recognizes single-line comments.
  A single line comment starts with the string `//` and extends to the end of the line.

The token `NUMBER` carries an integer or floating-point value. Strings have their own token payload.

The FsLex action maps reserved words to their token cases before returning an identifier.

When an identifier is read, we first have to check whether the identifier is one of our keywords.  If so, we assign the corresponding token type that is stored in the map `Keywords`.  Otherwise, the token type is set to `IDENTIFIER`.

FsYacc declares explicit token cases for all operators and punctuation.

White space, i.e. *space characters*, *tabulators*, and *carriage returns* are ignored.

Syntactically, newline characters are ignored. However, we still need to keep track of them in order to know the current line number, which is used for error messages.

Lex-buffer positions retain line and column information; `Parse.tokens` displays both.

Unknown characters stop scanning with a line/column diagnostic.

In [ ]:
let testScanner file =
    let source = System.IO.File.ReadAllText file
    printfn "%s" source
    printfn "%A" (Parse.tokens source)

In [ ]:
testScanner "Examples/sum-for.sl"

Below is the grammar for our language:
```
program
    : 𝜆  
    | stmnt program
    ;
stmnt 
    : FUNCTION IDENTIFIER '(' id_list ')' stmnt              # new
    | IF '(' bool_expr ')' stmnt                 
    | WHILE '(' bool_expr ')' stmnt
    | FOR '(' assign ';' bool_expr ';' assign ')' stmnt      # new
    | '{' program '}' 
    | assign ';'                                             # new
    | expr ';'
    | RETURN expr ';'                                        # new
    ;
assign                                                       # new
    : IDENTIFIER ':=' expr 
    ;
id_list                                                      # new
    : 𝜆      
    | ne_id_list
    ;
ne_id_list                                                   # new
    : IDENTIFIER
    | IDENTIFIER ',' ne_id_list
    ;
bool_expr                                                     # new
    : bool_expr OR conjunction                                # new
    | conjunction
    ;
conjunction                                                   # new
    : conjunction AND negation                                # new
    | negation                                                # new
    ;
negation                                                      # new
    : '!' negation                                            # new
    | bool_atom                                               # new
    ; 
bool_atom
    : '(' bool_expr ')'                                       # new
    | expr '==' expr     
    | expr '!=' expr     
    | expr '<=' expr     
    | expr '>=' expr     
    | expr '<'  expr      
    | expr '>'  expr     
    ;
expr: expr '+' product                 
    | expr '-' product
    | product
    ;          
product
    : product '*' factor               
    | product '/' factor
    | factor
    ;
factor
    : '(' expr ')' 
    | NUMBER
    | IDENTIFIER
    | IDENTIFIER '(' expr_list ')'
    ;
expr_list
    : 𝜆 
    | ne_expr_list
    ;
ne_expr_list
    : expr
    | expr ',' ne_expr_list
    ;
```

The *start variable* of our grammar is `program`.

The full FsYacc specification is shown in the preceding interpreter notebook and lives in `LanguageSyntax/Parser.fsy`. The `Define`, `For`, `Return`, `And`, `Or`, and `Not` actions are the exercise extensions.

An example program that conforms to this grammar is stored in the file `sum-for.sl` in the directory `Examples`.

In [ ]:
printfn "%s" (System.IO.File.ReadAllText "Examples/sum-for.sl")

A program is a list of statements.

**Exercise:** explain how the `For` AST records its initialization, test, update and body. Rewrite it into initialization followed by `While`; compare both programs using the same IO.

**Exercise:** explain why `And` and `Or` must short-circuit. Test `1 == 1 || missing == 0` without accessing the undefined variable.

The unified expression grammar places comparisons below arithmetic and above conjunction/disjunction. Parentheses can contain either arithmetic or boolean expressions.

The generated parser creates a typed statement tree.

In [ ]:
let parse = Interpreter.parse

`parse file` shares the AST visualization with the simple interpreter.

In [ ]:
printfn "%s" (System.IO.File.ReadAllText "Examples/factorial.sl")

In [ ]:
parse "Examples/factorial.sl" |> ignore

The exception `ReturnValue` carries a typed result. A call catches it while nested blocks and loops propagate it.

The following cell shows how F# catches the typed return exception.

In [ ]:
try raise (ReturnValue(Integer 42I)) with ReturnValue value -> printfn "%A" value

Execution and expression evaluation are mutually recursive, so they are defined in one `let rec`/`and` group.

Executing a block folds over its statements, returning the updated state. Definitions and variable values have separate maps.

`execute` updates variables and definitions, runs conditionals and loops, and raises `ReturnValue` for a return statement.

`evaluate` handles typed values, arithmetic, comparisons, short-circuit boolean operators, user-defined functions, and built-ins. Functions receive a fresh parameter environment; recursive calls reuse the definitions map. The numeric comparison helpers come from `Interpreter`; this frame retains the exercise language’s integer floor division, while the simple language’s `/` returns a real value.

In [ ]:
// Return is a typed control-flow exception; every enclosing block or loop propagates it.
let rec execute io state statement =
    match statement with
    | Block statements -> List.fold (execute io) state statements
    | Assign(name, expr) -> { state with Values = Map.add name (evaluate io state expr) state.Values }
    | Expression expr -> evaluate io state expr |> ignore; state
    | Define(name, parameters, body) ->
        if (Set.ofList parameters).Count <> parameters.Length then failwithf "Duplicate parameter in %s" name
        { state with Definitions = Map.add name (parameters, body) state.Definitions }
    | If(test, body) -> if truth (evaluate io state test) then execute io state body else state
    | While(test, body) ->
        let rec loop current =
            if truth (evaluate io current test) then loop (execute io current body) else current
        loop state
    | For(initial, test, update, body) ->
        let rec loop current =
            if truth (evaluate io current test) then loop (execute io (execute io current body) update) else current
        loop (execute io state initial)
    | Return expr -> raise (ReturnValue(evaluate io state expr))
and evaluate io state expr =
    match expr with
    | Literal value -> value
    | Variable name ->
        match Map.tryFind name state.Values with
        | Some value -> value
        | None -> failwithf "Undefined variable: %s" name
    | Not argument -> Boolean(not (truth (evaluate io state argument)))
    | Negative argument ->
        match evaluate io state argument with
        | Integer n -> Integer(-n) | Real n -> Real(-n)
        | _ -> failwith "Unary minus expects a number"
    | Binary(And, left, right) ->
        Boolean(truth (evaluate io state left) && truth (evaluate io state right))
    | Binary(Or, left, right) ->
        Boolean(truth (evaluate io state left) || truth (evaluate io state right))
    | Binary(op, left, right) ->
        let a, b = evaluate io state left, evaluate io state right
        match op with
        | Equal -> Boolean(equalValues a b) | NotEqual -> Boolean(not (equalValues a b))
        | Less -> Boolean(compareValues a b < 0) | Greater -> Boolean(compareValues a b > 0)
        | LessEqual -> Boolean(compareValues a b <= 0) | GreaterEqual -> Boolean(compareValues a b >= 0)
        | _ -> arithmetic op a b
    | Call(name, arguments) ->
        let values = List.map (evaluate io state) arguments
        match Map.tryFind name state.Definitions with
        | Some(parameters, body) ->
            if parameters.Length <> values.Length then
                failwithf "%s expects %d arguments, received %d" name parameters.Length values.Length
            let local = { state with Values = List.zip parameters values |> Map.ofList }
            try
                execute io local body |> ignore
                failwithf "Function %s completed without returning a value" name
            with ReturnValue value -> value
        | None ->
            match name, values with
            | "read", [] -> io.Read()
            | "print", args -> io.Write args; Integer BigInteger.Zero
            | name, [argument] when Map.containsKey name mathFunctions ->
                let result = mathFunctions[name] (asNumber argument)
                if not (System.Double.IsFinite result) then failwithf "Math domain or overflow error in %s" name
                Real result
            | _ -> failwithf "Unknown function or invalid argument count: %s" name

let run io source =
    try execute io emptyState (Parse.program source)
    with ReturnValue _ -> failwith "return is only valid inside a function"

let main file =
    let read () =
        printf "Please enter a number: "
        let text = System.Console.ReadLine()
        match BigInteger.TryParse(text, System.Globalization.NumberStyles.Integer, System.Globalization.CultureInfo.InvariantCulture) with
        | true, integer -> Integer integer
        | _ -> Real(System.Double.Parse(text, System.Globalization.CultureInfo.InvariantCulture))
    let io = { Read = read; Write = (List.map Interpreter.valueText >> String.concat " " >> printfn "%s") }
    run io (System.IO.File.ReadAllText file)

In [ ]:
printfn "%s" (System.IO.File.ReadAllText "Examples/factorial.sl")

In [ ]:
let demonstrate file input =
    let output = ResizeArray<Value list>()
    let io = { Read = (fun () -> input); Write = output.Add }
    let state = run io (System.IO.File.ReadAllText file)
    output |> Seq.iter (List.map Interpreter.valueText >> String.concat " " >> printfn "%s")
    state, List.ofSeq output

In [ ]:
let state, factorialOutput = demonstrate "Examples/factorial.sl" (Integer 10I)
checkNotebook (factorialOutput.Length = 24)
checkNotebook (factorialOutput |> List.last |> List.last = Integer 15511210043330985984000000I)

In [ ]:
let solved, _ = demonstrate "Examples/solve.sl" (Integer 10I)
checkNotebook (abs(asNumber solved.Values["x"] - 0.7390851332151607) < 1e-12)

In [ ]:
let _, piOutput = demonstrate "Examples/pi.sl" (Integer 10I)
checkNotebook (abs(asNumber (piOutput |> List.last |> List.last) - System.Math.PI) < 1e-14)

In [ ]:
let _, sumOutput = demonstrate "Examples/sum-for.sl" (Integer 10I)
checkNotebook (sumOutput |> List.last |> List.last = Integer 5050I)

In [ ]:
let _, eOutput = demonstrate "Examples/e.sl" (Integer 10I)
checkNotebook (abs(asNumber (eOutput |> List.last |> List.last) - exp 1.0) < 1e-14)

In [ ]:
checkNotebook (floorDivide -7I 3I = -3I)
let shortCircuit = run { Read = (fun () -> Integer 0I); Write = ignore } "x := 1 == 1 || missing == 0;"
checkNotebook (shortCircuit.Values["x"] = Boolean true)

The original `power.sl`, `primes.sl`, and `carmichael.sl` files use a set-language extension absent from this notebook’s grammar. Their complete algorithms are translated separately as `Examples/power.fsx`, `Examples/primes.fsx`, and `Examples/carmichael.fsx`. The `.sl` fixtures are retained as lecture input data; the F# scripts can be run with `dotnet fsi`.